# Validation Test Evaluation - Static droplet on slip wall (transient simulation)

This notebook and the corresponding simulation setup notebook (StaticDropletOnSlipWall_transient_Run.ipynb) are part of published results (cf. 5.1.3) found in *The extended discontinuous Galerkin method adapted for moving contact line problems via the generalized Navier boundary condition* (https://doi.org/10.1002/fld.5016).

In [ ]:
#r "BoSSSpad.dll"
//#r "..\..\..\src\L4-application\BoSSSpad\bin\Release\net8.0\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [ ]:
wmg.Init("CLpaper_StaticDropletOnSlipWall");
//OpenOrCreateDatabase(@"\\fdygitrunner\ValidationTests\databases\CLpaper_StaticDropletOnSlipWall");

Opening existing database '\\fdygitrunner\ValidationTests\databases\CLpaper_StaticDropletOnSlipWall'.


In [3]:
using System.IO;
using BoSSS.Foundation.Quadrature;
using BoSSS.Solution.XNSECommon;
using BoSSS.Application.XNSE_Solver.Logging;
using BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases;

## Sessions

In [ ]:
static var sessions = wmg.Sessions;
//static var sessions = databases.Pick(0).Sessions;
sessions

#0: CLpaper_StaticDropletOnSlipWall	StaticDropletOnWall_transient_meshStudy_mesh60	11/26/2025 1:25:07 PM	307c0224...
#1: CLpaper_StaticDropletOnSlipWall	StaticDropletOnWall_transient_meshStudy_mesh40	11/26/2025 1:24:55 PM	5d16607d...
#2: CLpaper_StaticDropletOnSlipWall	StaticDropletOnWall_transient_meshStudy_mesh20	11/26/2025 1:24:44 PM	5f2881a5...


## Spurious velocities — Transient simulations

In [5]:
int[] Meshes = { 20, 40, 60 };

In [ ]:
// origin database including all sessions, in case of missing sessions in wmg, due to rerunning only a subset of all needed runs (reduced computational cost)
var originDB = OpenOrCreateDatabase(@"\\fdygitrunner\ValidationTests\databases\bkup-2025Oct29_000000.CLpaper_StaticDropletOnSlipWall");

In [6]:
List<ISessionInfo> evalSess = new List<ISessionInfo>();

In [ ]:
foreach (int mesh in Meshes) {
    string studyName = $"StaticDropletOnWall_transient_meshStudy_mesh{mesh}";
    Console.WriteLine("Searching for: " + studyName);
    var studySess = sessions.Where(sess => sess.Name.Contains(studyName));
    int studyCount = studySess.Count();
    if (studyCount == 0) {
        //Console.WriteLine("Not found!");
        
        // try to get from originDB
        var originSess = originDB.Sessions.Where(sess => sess.Name.Contains(studyName));
        int originCount = originSess.Count();
        if (originCount == 0) {
            Console.WriteLine("Not found!");
        } else if (originCount > 1) {
            Console.WriteLine("Restart session found in originDB! Take last run");       
            evalSess.Add(originSess.Where(sess => sess.Name.ToLower().EndsWith($"_restart{originCount-1}")).Single());
        } else {
            evalSess.Add(originSess.Single());
            Console.WriteLine("found in originDB");
        }

    } else if (studyCount > 1) {
        Console.WriteLine("Restart session found! Take last run");       
        evalSess.Add(studySess.Where(sess => sess.Name.ToLower().EndsWith($"_restart{studyCount-1}")).Single());
    } else {
        evalSess.Add(studySess.Single());
        Console.WriteLine("found");
    }
}
evalSess

Searching for: StaticDropletOnWall_transient_meshStudy_mesh20
found
Searching for: StaticDropletOnWall_transient_meshStudy_mesh40
found
Searching for: StaticDropletOnWall_transient_meshStudy_mesh60
found


#0: CLpaper_StaticDropletOnSlipWall	StaticDropletOnWall_transient_meshStudy_mesh20	11/26/2025 1:24:44 PM	5f2881a5...
#1: CLpaper_StaticDropletOnSlipWall	StaticDropletOnWall_transient_meshStudy_mesh40	11/26/2025 1:24:55 PM	5d16607d...
#2: CLpaper_StaticDropletOnSlipWall	StaticDropletOnWall_transient_meshStudy_mesh60	11/26/2025 1:25:07 PM	307c0224...


In [8]:
NUnit.Framework.Assert.AreEqual(3, evalSess.Count, $"Not enough sessions for evaluation.");

### compute times

In [9]:
foreach (var sess in evalSess) {
    var nameData = sess.Name.Split(new string[] { "_" }, StringSplitOptions.RemoveEmptyEntries);
    string sessName = "";
    int nameLength = nameData.Length;
    for (int i = 0; i < nameLength; i++) {
        if (nameData[i].StartsWith("restart"))
            continue;
        if (i == 0)
            sessName += nameData[i];
        else    
            sessName += "_" + nameData[i];
    }

    // determine compute time
    Stack<ISessionInfo>  procSIs = new Stack<ISessionInfo>();
    procSIs.Push(sess);
    var currSI = sess;
    var currTimesteps = currSI.Timesteps.OrderBy(ts => ts.TimeStepNumber.MajorNumber);
    int timesteps = currTimesteps.Last().TimeStepNumber.MajorNumber;
    double physicalTime = currTimesteps.Last().PhysicalTime;
    TimeSpan computeTime = currTimesteps.Last().CreationTime - currSI.Timesteps.First().CreationTime;
    var rSIs = sessions.Where(sess => sess.ID.Equals(currSI.RestartedFrom));
    while(!rSIs.IsNullOrEmpty()) {
        var rSI = rSIs.Single();
        procSIs.Push(rSI);
        currSI = rSI;
        currTimesteps = currSI.Timesteps.OrderBy(ts => ts.TimeStepNumber.MajorNumber);
        computeTime = computeTime + (currTimesteps.Last().CreationTime - currTimesteps.First().CreationTime);
        rSIs = sessions.Where(sess => sess.ID.Equals(currSI.RestartedFrom));
    }

    Console.WriteLine($"Session - {sessName}: timesteps = {timesteps} (t_end = {physicalTime}); compute time = {computeTime.Days}:{computeTime.Hours}");
}

Session - StaticDropletOnWall_transient_meshStudy_mesh20: timesteps = 12500 (t_end = 125.00000000002704); compute time = 0:13
Session - StaticDropletOnWall_transient_meshStudy_mesh40: timesteps = 12500 (t_end = 125.00000000002704); compute time = 1:16
Session - StaticDropletOnWall_transient_meshStudy_mesh60: timesteps = 12500 (t_end = 125.00000000002704); compute time = 3:11


### Evaluate terminal time step

In [10]:
public static void EvaluateTerminalTimeStep(ISessionInfo sess, double refL2velocity, double refL2pressure) {

    string[] nameData = sess.Name.Split(new string[] { "_" }, StringSplitOptions.RemoveEmptyEntries);
    string meshName;
    int nameLength = nameData.Length;
    if (nameData[nameLength - 1].ToLower().StartsWith("restart"))
        meshName = nameData[nameLength - 2];
    else    
        meshName = nameData.Last();
   

    var terminalStep = sess.Timesteps.OrderBy(ts => ts.TimeStepNumber.MajorNumber).Last();

    Console.WriteLine($"{meshName}: time = {terminalStep.PhysicalTime}");

    // L2 error velocity
    double uL2 = terminalStep.GetField("VelocityX").L2Norm();
    double vL2 = terminalStep.GetField("VelocityY").L2Norm();
    double velL2 = (uL2.Pow2()+vL2.Pow2()).Sqrt();
    Console.WriteLine($"L2-error velocity: {velL2}");
    NUnit.Framework.Assert.LessOrEqual(velL2, refL2velocity, 
                $"L2-error for velocity does not match reference value {refL2velocity}.");

    // =======================================
    DGField phi = terminalStep.GetField("Phi");
    LevelSet LevSet = new LevelSet(phi.Basis, "LevelSet"); 
    LevSet.Acc(1.0, phi); 
    LevelSetTracker LsTrk = new LevelSetTracker((BoSSS.Foundation.Grid.Classic.GridData) phi.GridDat, CutCellQuadratureMethod.Saye, 1, new string[] { "A", "B" }, LevSet);
    LsTrk.UpdateTracker(125.0);

    // get pressure niveau outside droplet
    DGField pressure = terminalStep.GetField("Pressure");
    int order        = 8;
    var SchemeHelper = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS.ToArray(), order, 1).XQuadSchemeHelper;
    SpeciesId spcBId = LsTrk.SpeciesIdS[1];
    var vqs = SchemeHelper.GetVolumeQuadScheme(spcBId);
    double areaB     = XNSEUtils.GetSpeciesArea(LsTrk, spcBId);
    double p0        = 0.0;
    CellQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat, vqs.Compile(LsTrk.GridDat, order),
        delegate (int i0, int Length, QuadRule QR, MultidimensionalArray EvalResult) {
            for (int i = 0; i < Length; i++) {
                double p0i = ((XDGField)pressure).GetSpeciesShadowField("B").GetMeanValue(i0+i);
                for (int k = 0; k < QR.NoOfNodes; k++) {
                    EvalResult[i,k,0] = p0i;
                }
            }
        },
        delegate (int i0, int Length, MultidimensionalArray ResultsOfIntegration) {
            for (int i = 0; i < Length; i++) {
                p0 += ResultsOfIntegration[i, 0]/areaB;
            }
        }
    ).Execute();

    // L2 error outside droplet (spceciesB)
    var pOutN = ((XDGField)pressure).GetSpeciesShadowField("B");
    pOutN.AccConstant(-p0);
    double pOutL2 = pOutN.L2Error((ScalarFunction)null, vqs);

    // L2 error inside droplet (spceciesA)
    SpeciesId spcAId = LsTrk.SpeciesIdS[0];
    vqs   = SchemeHelper.GetVolumeQuadScheme(spcAId);
    double sigma = 1.0;
    double r     = 0.25;
    Func<double[], double> refA = (X => sigma/r);
    var pInN     = ((XDGField)pressure).GetSpeciesShadowField("A");
    pInN.AccConstant(-p0);
    double pInL2 = pInN.L2Error(refA.Vectorize(), vqs);

    // L2 error pressure
    double pL2 = (pInL2.Pow2()+pOutL2.Pow2()).Sqrt();  
    Console.WriteLine($"L2-error pressure: {pL2}");
    NUnit.Framework.Assert.LessOrEqual(pL2, refL2pressure, 
                $"L2-error for pressure does not match reference value {refL2pressure}.");

}

### TABLE 2 - L2-error norms of spurious velocities and Laplace-Young equation for the terminal time step at t = 125

L2-error norms for mesh 20

In [ ]:
EvaluateTerminalTimeStep(evalSess.Pick(0), 1.85e-6, 1.24e-3)

mesh20: time = 125.00000000002704
L2-error velocity: 1.6243995379547537E-06
L2-error pressure: 0.0012342827318692217


L2-error norms for mesh 40

In [ ]:
EvaluateTerminalTimeStep(evalSess.Pick(1), 4.52e-7, 4.63e-4)

mesh40: time = 125.00000000002704
L2-error velocity: 2.0626049272332586E-07
L2-error pressure: 0.0004628570840873432


L2-error norms for mesh 60

In [18]:
//EvaluateTerminalTimeStep(evalSess.Pick(2), 3.21e-7, 5.43e-4)
EvaluateTerminalTimeStep(evalSess.Pick(2), 6.67e-7, 1.40e-3)

mesh60: time = 125.00000000002704
L2-error velocity: 6.663381270937029E-07
L2-error pressure: 0.00139162029066343


### Evaluate temporal evolution

In [19]:
public static List<Plot2Ddata> EvaluateData(this List<ISessionInfo> procSess) {
    
    List<Plot2Ddata> plotData = new List<Plot2Ddata>();

    int numberSessions = procSess.Count();

    string[] values = { "spurious velocities", "interface changerate" };
    int numberValues = values.Length;
    //for (int vIdx = 0; vIdx < values.Length; vIdx++) {

        double[][] times = new double[numberSessions][];
        double[][][] valueDatas = new double[numberSessions][][];

        // Read all data
        for (int j = 0; j < numberSessions; j++) {

            var sess = sessions.ElementAt(j);
            Console.WriteLine("Processing session: " + sess.Name);

            // Get session history
            List<ISessionInfo> restartSessionList = new List<ISessionInfo>();
            restartSessionList.Add(sess);
            Guid restartID = sess.RestartedFrom;

            while(restartID != Guid.Empty) {
                try {
                    var rSess = sessions.Where(sess => sess.ID == restartID).Single();
                    Console.WriteLine("  Found restart session: " + rSess.Name);
                    restartSessionList.Add(rSess);
                    restartID = rSess.RestartedFrom;

                } catch { // do nothing if session not found
                    restartID = Guid.Empty;
                }
            }
            restartSessionList.Reverse();

            Console.WriteLine("  Merging data from " + restartSessionList.Count + " sessions.");

            List<double> time = new List<double>();
            List<double>[] valueData = new List<double>[numberValues];
            for(int vIdx = 0; vIdx < numberValues; vIdx++) {
                valueData[vIdx] = new List<double>();
            }

            for(int rSi = 0; rSi < restartSessionList.Count(); rSi++) {

                var timesteps = restartSessionList.ElementAt(rSi).Timesteps;
            
                for (int i = 0; i < timesteps.Count(); i++) {

                    var ts = timesteps.Pick(i);
                    double l_time = ts.PhysicalTime;
                    double[] l_values = new double[numberValues];
                
                    DGField phi = ts.GetField("Phi");
                    LevelSet LevSet = new LevelSet(phi.Basis, "LevelSet"); 
                    LevSet.Acc(1.0, phi);           
                    LevelSetTracker LsTrk = new LevelSetTracker((BoSSS.Foundation.Grid.Classic.GridData) phi.GridDat, CutCellQuadratureMethod.Saye, 1, new string[] { "A", "B" }, LevSet);
                    LsTrk.UpdateTracker(ts.PhysicalTime);  
         
                    XDGField VelX = (XDGField)ts.GetField("VelocityX");
                    XDGField VelY = (XDGField)ts.GetField("VelocityY");
                    XDGField[] Velocity = new XDGField[] {VelX, VelY};  

                    // get spurious velocities 
                    {
                        double spuriousVelocity = 0.0;                        

                        int momentFittingOrder  = 6;
                        var SchemeHelper = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS.ToArray(), momentFittingOrder, 1).XQuadSchemeHelper;
                                    
                        for(int iSpc = 0; iSpc < LsTrk.SpeciesIdS.Count; iSpc++) {
                            SpeciesId spId = LsTrk.SpeciesIdS[iSpc];

                            var scheme = SchemeHelper.GetVolumeQuadScheme(spId);

                            int D = LsTrk.GridDat.SpatialDimension;
                            for(int d = 0; d < D; d++) {
                                DGField U = Velocity.ElementAt(d);
                                spuriousVelocity += U.L2Error(null, momentFittingOrder, scheme);
            
                            }
                        } 
                        l_values[0] = spuriousVelocity;
                    } 
 
                    // get surface divergence
                    {
                        ConventionalDGField[] meanVelocity = XNSEUtils.GetMeanVelocity(Velocity, LsTrk, 1.0, 1.0);
                        double surfaceDivergence = EnergyUtils.GetSurfaceChangerate(LsTrk, meanVelocity, 6);   
                        l_values[1] = surfaceDivergence;
                    } 

                    if(time.IsNullOrEmpty() || l_time > time.Last()) {
                        time.Add(l_time);
                        for(int vIdx = 0; vIdx < numberValues; vIdx++) {
                            valueData[vIdx].Add(l_values[vIdx]);
                        }
                    }

                }
            }

            Console.WriteLine("... Done");
                            
            times[j] = time.ToArray();
            valueDatas[j] = new double[numberValues][];
            for(int vIdx = 0; vIdx < numberValues; vIdx++) {
                valueDatas[j][vIdx] = valueData[vIdx].ToArray();
            }
        
        }

        // Build DataSets
        Console.WriteLine("Build DataSets");
        for(int vIdx = 0; vIdx < numberValues; vIdx++) {

            KeyValuePair<string, double[][]>[] dataRowsValue = new KeyValuePair<string, double[][]>[numberSessions];
            for(int i = 0; i < numberSessions; i++) {
                string sessName = (procSess.Pick(i).Name).Replace("_", "-");
                dataRowsValue[i] = new KeyValuePair<string, double[][]>(sessName, new double[][] { times[i], valueDatas[i][vIdx] });
            }
            Console.WriteLine("Element at {0}: time vs {1}", vIdx, values[vIdx]);
            Plot2Ddata plt = new Plot2Ddata(dataRowsValue);
            plt.Xlabel = "time";
            plt.Ylabel = values[vIdx];
            plotData.Add(plt);
        }

    return plotData;
}

In [20]:
var evalData = evalSess.EvaluateData();

Processing session: StaticDropletOnWall_transient_meshStudy_mesh60
  Merging data from 1 sessions.
... Done
Processing session: StaticDropletOnWall_transient_meshStudy_mesh40
  Merging data from 1 sessions.
... Done
Processing session: StaticDropletOnWall_transient_meshStudy_mesh20
  Merging data from 1 sessions.
... Done
Build DataSets
Element at 0: time vs spurious velocities
Element at 1: time vs interface changerate


In [21]:
public static Plot2Ddata GetEvalDataPlot(List<Plot2Ddata> data, int dataIndex, string[] meshSizes) {

    Plot2Ddata plt =  new Plot2Ddata();
    plt.Xlabel = "time";
    plt.Ylabel = data.ElementAt(dataIndex).Ylabel;
    
    var datGroups = data.ElementAt(dataIndex).dataGroups;
    int lcIdx = 1;
    for (int i = 0; i < datGroups.Count(); i++) {

        var fmt = new PlotFormat();
        fmt.Style = Styles.Lines; 
        fmt.DashType = DashTypes.Solid;
        fmt.LineWidth = 2;
        fmt.LineColor = (LineColors)(lcIdx); //LineColors.Blue;

        string[] nameData = datGroups[i].Name.Split(new string[] { "-" }, StringSplitOptions.RemoveEmptyEntries);
        if (meshSizes.Contains(nameData[3])) {
            plt.AddDataGroup(nameData[3], datGroups[i].Abscissas, datGroups[i].Values, fmt);
            lcIdx++;
        }
    }

    plt.XrangeMin = 0.0;
    plt.XrangeMax = 125.0;
    plt.ShowLegend = true;

    return plt;

} 

### FIGURE 6 

In [22]:
Plot2Ddata[,] Fig6 = new Plot2Ddata[1, 2];

string[] meshStudy = new string[] { "mesh20", "mesh40", "mesh60" };
Fig6[0, 0] = GetEvalDataPlot(evalData, 0, meshStudy);
Fig6[0, 1] = GetEvalDataPlot(evalData, 1, meshStudy);

Fig6.ToGnuplot().PlotSVG(xRes:1300,yRes:500)

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 5x10 -6 
 
 
 
 
 1x10 -5 
 
 
 
 
 1.5x10 -5 
 
 
 
 
 2x10 -5 
 
 
 
 
 2.5x10 -5 
 
 
 
 
 3x10 -5 
 
 
 
 
 3.5x10 -5 
 
 
 
 
 4x10 -5 
 
 
 
 
 0 
 
 
 
 
 20 
 
 
 
 
 40 
 
 
 
 
 60 
 
 
 
 
 80 
 
 
 
 
 100 
 
 
 
 
 120 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 spurious velocities 
 
 
 
 
 time 
 
 
 
 
 mesh20 
 
 
 
 
 mesh20 
 
 
 
	<path stroke='rgb(255, 0, 0)' d='M454.2,62.1 L507.6,62.1 M111.5,437.5 L116.2,413.8 L116.3,413.8 L121.1,418.4 L121.1,418.5 L125.9,421.8
 L126.0,421.8 L130.7,423.6 L130.7,423.7 L130.8,423.7 L135.5,426.6 L135.6,426.6 L140.3,428.2 L140.4,428.2
 L145.2,429.2 L150.0,430.2 L150.1,430.2 L154.8,431.1 L154.9,431.1 L159.6,431.6 L159.7,431.6 L164.5,432.0
 L169.3,432.3 L169.4,432.3 L174.1,432.3 L174.2,432.3 L178.9,432.4 L179.0,432.4 L183.7,432.5 L183.8,432.5
 L188.6,432.5 L193.4,432.5 L193.5,432.5 L198.2,432.5 L198.3,432.5 L203.0,432.5 L203.1,432.5 L207.8,432.5
 L207.9,432.5 L212.7,432.6 L217.5,432.6 L217.6,432.6 L222.3,432.6 L222.4,432.6 L227.1,432.6 L227.2,432.6
 L231.9,432.5 L232.0,432.5 L236.8,432.5 L241.6,432.5 L241.7,432.5 L246.4,432.4 L246.5,432.4 L251.2,432.3
 L251.3,432.3 L256.1,432.3 L260.9,432.2 L261.0,432.2 L265.7,432.1 L265.8,432.1 L270.5,432.0 L270.6,432.0
 L275.3,432.0 L275.4,432.0 L280.2,431.9 L285.0,431.8 L285.1,431.8 L289.8,431.7 L289.9,431.7 L294.6,431.6
 L294.7,431.6 L299.4,431.5 L299.5,431.5 L304.3,431.4 L309.1,431.3 L309.2,431.3 L313.9,431.2 L314.0,431.2
 L318.7,431.1 L318.8,431.1 L323.5,431.0 L323.6,431.0 L328.4,430.9 L333.2,430.8 L333.3,430.8 L338.0,430.7
 L338.1,430.7 L342.8,430.6 L342.9,430.6 L347.7,430.5 L352.5,430.4 L357.3,430.3 L357.4,430.3 L362.1,430.2
 L362.2,430.2 L366.9,430.1 L367.0,430.1 L371.8,430.0 L376.6,429.9 L376.7,429.9 L381.4,429.8 L381.5,429.8
 L386.2,429.7 L386.3,429.7 L391.0,429.5 L391.1,429.5 L395.9,429.4 L400.7,429.3 L400.8,429.3 L405.5,429.2
 L405.6,429.2 L410.3,429.1 L410.4,429.1 L415.1,429.0 L415.2,429.0 L420.0,428.8 L424.8,428.7 L424.9,428.7
 L429.6,428.6 L429.7,428.6 L434.4,428.5 L434.5,428.5 L439.3,428.3 L444.1,428.2 L448.9,428.1 L449.0,428.1
 L453.7,428.0 L453.8,428.0 L458.5,427.8 L458.6,427.8 L463.4,427.7 L468.2,427.6 L468.3,427.6 L473.0,427.5
 L473.1,427.5 L477.8,427.3 L477.9,427.3 L482.6,427.2 L482.7,427.2 L487.5,427.1 L492.3,426.9 L492.4,426.9
 L497.1,426.8 L497.2,426.8 L501.9,426.6 L502.0,426.6 L506.7,426.5 L506.8,426.5 L511.6,426.4 L516.4,426.2
 L516.5,426.2 L521.2,426.1 L521.3,426.1 L526.0,426.0 L526.1,426.0 L530.8,425.9 L530.9,425.9 L535.7,425.8
 L540.5,425.7 L540.6,425.7 L545.3,425.5 L545.4,425.5 L550.1,425.4 L550.2,425.4 L555.0,425.3 L559.8,425.1
 L559.9,425.1 L564.6,425.0 L564.7,425.0 L569.4,424.9 L569.5,424.9 L574.2,424.7 L574.3,424.7 L579.1,424.6
 L583.9,424.5 L584.0,424.5 L588.7,424.3 L588.8,424.3 L593.5,424.2 L593.6,424.2 '/> 
 
 mesh40 
 
 
 mesh40 
 
 
 
	<path stroke='rgb( 0, 255, 0)' d='M454.2,86.1 L507.6,86.1 M111.5,437.5 L116.2,383.0 L116.3,383.1 L116.3,383.2 L121.1,404.1 L121.1,404.2
 L121.1,404.3 L125.9,418.4 L126.0,418.4 L130.7,415.9 L130.8,415.9 L135.5,423.5 L135.6,423.6 L140.3,427.0
 L140.4,426.9 L145.2,425.2 L150.0,426.0 L150.1,426.0 L154.8,429.6 L154.9,429.6 L154.9,429.7 L159.6,430.3
 L159.7,430.2 L164.5,430.7 L169.3,430.9 L169.4,430.9 L174.1,430.3 L174.2,430.3 L178.9,430.3 L179.0,430.3
 L183.7,430.7 L183.8,430.7 L188.6,430.8 L193.4,430.7 L193.5,430.7 L198.2,430.8 L198.3,430.8 L203.0,431.0
 L203.1,431.0 L207.8,431.4 L207.9,431.4 L212.7,431.4 L217.5,431.5 L217.6,431.5 L222.3,431.6 L222.4,431.6
 L227.1,431.7 L227.2,431.7 L231.9,431.9 L232.0,431.9 L236.8,432.0 L241.6,432.1 L241.7,432.1 L246.4,4